In [ ]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

if Path('/kaggle/working').is_dir():
    provider = 'kaggle'
    output_dir = Path('/kaggle/working')
elif Path('/content').is_dir():
    provider = 'colab-compatible'
    output_dir = Path('/content')
else:
    raise RuntimeError('Use the remote Kaggle/Colab runtime, not the laptop.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU unavailable. Check the free accelerator setting; no CPU fallback.')

devices = []
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    devices.append({'index': index, 'name': props.name,
                    'vram_gib': round(props.total_memory / 2**30, 3)})

# Verify a real CUDA operation; multiple GPUs are not pooled as one VRAM space.
with torch.inference_mode():
    probe = torch.ones((8, 8), device='cuda:0')
    assert torch.all(probe @ probe == 8).item()
del probe
torch.cuda.empty_cache()

report = {'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
          'provider': provider, 'python': platform.python_version(),
          'torch': torch.__version__, 'cuda_runtime': torch.version.cuda,
          'devices': devices, 'gpu_operation_verified': True,
          'model_loaded': False, 'training_run': False,
          'account_quota': 'check_provider_UI_separately'}
assert devices and all(device['vram_gib'] > 0 for device in devices)
target = output_dir / ('denim_gpu_check_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
target.write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
assert json.loads(target.read_text(encoding='utf-8'))['gpu_operation_verified'] is True
print(json.dumps(report, indent=2))
print('Report:', target)


In [ ]:
"""Remote-only model loading check; synthetic inputs do not evaluate denim retrieval."""

import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import subprocess
import sys
import time
from datetime import datetime, timezone

if Path('/kaggle/working').is_dir():
    output_dir = Path('/kaggle/working')
elif Path('/content').is_dir():
    output_dir = Path('/content')
else:
    raise RuntimeError('Run on the remote GPU, not the laptop.')

import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable; no CPU fallback.')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
     'open_clip_torch==3.3.0'], check=True,
)

import open_clip
from huggingface_hub import HfApi, snapshot_download
from PIL import Image

repo_id = 'Marqo/marqo-fashionSigLIP'
# Resolve this published commit once; every downloaded file uses its full SHA.
revision = HfApi().model_info(repo_id, revision='c56244c').sha
assert revision and len(revision) == 40
print('Model revision:', revision, flush=True)
snapshot = Path(snapshot_download(
    repo_id, revision=revision,
    allow_patterns=['open_clip_config.json', 'open_clip_model.safetensors'],
    cache_dir=str(output_dir / 'hf_cache'),
))
weights = snapshot / 'open_clip_model.safetensors'
assert weights.is_file() and weights.stat().st_size > 0
with weights.open('rb') as stream:
    weight_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
config = json.loads((snapshot / 'open_clip_config.json').read_text())
assert config['model_cfg']['embed_dim'] == 768
assert config['model_cfg']['vision_cfg']['image_size'] == 224

model, _, preprocess = open_clip.create_model_and_transforms(
    'local-dir:' + str(snapshot), device='cuda:0', require_pretrained=True,
)
model.eval()
# No external image upload is needed to check the tensor and model execution path.
batch = torch.stack([
    preprocess(Image.new('RGB', (224, 224), color))
    for color in [(40, 65, 100), (90, 110, 135)]
]).to('cuda:0')
assert batch.shape == (2, 3, 224, 224)
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    model.encode_image(batch, normalize=True)  # warm-up
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    embeddings = model.encode_image(batch, normalize=True).float()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
assert embeddings.shape == (2, 768) and torch.isfinite(embeddings).all().item()
assert torch.allclose(embeddings.norm(dim=-1), torch.ones(2, device='cuda:0'), atol=0.01)

report = {
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    'repo_id': repo_id, 'revision': revision, 'weights_sha256': weight_sha256,
    'python': platform.python_version(),
    'packages': {name: importlib.metadata.version(name) for name in
                 ['torch', 'torchvision', 'open_clip_torch', 'timm', 'huggingface_hub', 'safetensors', 'Pillow']},
    'cuda_runtime': torch.version.cuda, 'device': torch.cuda.get_device_name(0),
    'batch_shape': list(batch.shape), 'embedding_shape': list(embeddings.shape),
    'forward_seconds_after_warmup': elapsed,
    'peak_allocated_gib': torch.cuda.max_memory_allocated() / 2**30,
    'input': 'two synthetic solid-color images; not a retrieval evaluation',
    'model_loaded': True, 'training_run': False, 'denim_quality_evaluated': False,
}
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
target = output_dir / ('denim_model_check_' + stamp + '.json')
target.write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
assert json.loads(target.read_text())['model_loaded'] is True
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True)
(output_dir / ('denim_environment_' + stamp + '.txt')).write_text(freeze.stdout, encoding='utf-8')
print(json.dumps(report, indent=2))
print('Report:', target)


In [ ]:
"""Run after check_remote_model.py in the same remote kernel; synthetic smoke check."""

import copy

assert torch.cuda.is_available() and report['model_loaded']
assert report['denim_quality_evaluated'] is False
torch.manual_seed(17)
model.requires_grad_(False)
model.visual.requires_grad_(True)
model.visual.train()
optimizer = torch.optim.AdamW(model.visual.parameters(), lr=1e-5)
scaler = torch.amp.GradScaler('cuda')
optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
with torch.autocast('cuda', dtype=torch.float16):
    features = model.encode_image(batch, normalize=True).float()
    # Arbitrary synthetic objective checks gradients, not denim relevance learning.
    loss = (features[0] - features[1]).square().mean()
assert torch.isfinite(loss).item()
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
gradients = [p.grad for p in model.visual.parameters() if p.grad is not None]
assert gradients and all(torch.isfinite(g).all().item() for g in gradients)
assert any(torch.count_nonzero(g).item() for g in gradients)
scaler.step(optimizer)
scaler.update()
torch.cuda.synchronize()
step_seconds = time.perf_counter() - started
peak_gib = torch.cuda.max_memory_allocated() / 2**30
optimizer.zero_grad(set_to_none=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
checkpoint = output_dir / ('denim_synthetic_smoke_' + stamp + '.pt')
torch.save({
    'visual': model.visual.state_dict(), 'optimizer': optimizer.state_dict(),
    'scaler': scaler.state_dict(), 'torch_rng': torch.get_rng_state(),
    'cuda_rng': torch.cuda.get_rng_state_all(), 'step': 1,
    'model_revision': revision, 'purpose': 'synthetic smoke check only',
}, checkpoint)
# Deliberately change a parameter and LR, then verify actual restoration.
parameter = next(model.visual.parameters())
expected = parameter.detach().clone()
optimizer_lr = optimizer.param_groups[0]['lr']
scaler_state = copy.deepcopy(scaler.state_dict())
with torch.no_grad():
    parameter.zero_()
optimizer.param_groups[0]['lr'] = 0.1
saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
model.visual.load_state_dict(saved['visual'], strict=True)
optimizer.load_state_dict(saved['optimizer'])
scaler.load_state_dict(saved['scaler'])
torch.set_rng_state(saved['torch_rng'])
torch.cuda.set_rng_state_all(saved['cuda_rng'])
assert torch.equal(parameter, expected)
assert optimizer.param_groups[0]['lr'] == optimizer_lr
assert scaler.state_dict() == scaler_state
assert saved['step'] == 1 and saved['model_revision'] == revision
assert torch.equal(torch.get_rng_state(), saved['torch_rng'])

backward_report = {
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    'revision': revision, 'device': torch.cuda.get_device_name(0),
    'batch_shape': list(batch.shape), 'precision': 'fp32 parameters, fp16 autocast and GradScaler',
    'trainable_parameters': sum(p.numel() for p in model.visual.parameters()),
    'synthetic_loss': loss.item(), 'step_seconds': step_seconds,
    'peak_allocated_gib': peak_gib,
    'backward_verified': True, 'checkpoint_roundtrip_verified': True,
    'checkpoint_bytes': checkpoint.stat().st_size,
    'checkpoint': checkpoint.name,
    'resume_scope': 'same-kernel visual weights, AdamW, scaler, torch RNG; not interrupted-job determinism',
    'denim_training_run': False, 'denim_quality_evaluated': False,
}
target = output_dir / ('denim_backward_check_' + stamp + '.json')
target.write_text(json.dumps(backward_report, indent=2) + chr(10), encoding='utf-8')
assert json.loads(target.read_text())['checkpoint_roundtrip_verified'] is True
print(json.dumps(backward_report, indent=2))
print('Report:', target)
